# 01 - Preprocesamiento de Brent, GPR y mercado mundial

## Objetivo

Construir un dataset mensual integrado para estudiar si el riesgo geopolítico y la información de oferta y demanda mundial ayudan a anticipar el precio del Brent del mes siguiente. El período de trabajo es **enero de 1998 a diciembre de 2025**; no se fabrican observaciones ni se interpolan meses.

La historia se organiza en tres fuentes: precio Brent de EIA, índices GPR de Caldara e Iacoviello y mercado mundial de EIA-STEO. Este notebook inspecciona, limpia, valida unidades, integra y exporta el archivo que utilizan los notebooks 02 y 03.

## 1. Fuentes y decisiones

- **Brent:** Europe Brent Spot Price FOB, promedio mensual, USD por barril ([EIA](https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?f=m&n=PET&s=RBRTE)).
- **GPR:** índice mensual de riesgo geopolítico, amenazas (`GPRT`) y actos (`GPRA`) de Caldara e Iacoviello ([sitio oficial](https://www.matteoiacoviello.com/gpr.htm)).
- **Mercado mundial:** STEO Data Browser de EIA. `PAPR_WORLD` y `PATC_WORLD` miden petróleo y otros líquidos; `COPR_WORLD` mide solo crudo. Todas están expresadas en millones de barriles por día.

El balance se calcula como `PAPR_WORLD - PATC_WORLD`, porque producción y consumo pertenecen a la misma categoría. `COPR_WORLD` se conserva por separado y no se resta del consumo total. El archivo STEO es un vintage revisable: se excluyen 2026-2027 y los valores 1998-2025 se describen como historia revisada disponible en ese vintage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

BRENT_FILE = RAW / "brent_eia.xls"
GPR_FILE = RAW / "gpr_monthly.xls"
STEO_FILE = RAW / "3a._International_Petroleum_and_Other_Liquids_Production_Consumption_and_Inventories.csv"
OUTPUT_FILE = PROCESSED / "brent_gpr_market_monthly.csv"

START = pd.Timestamp("1998-01-01")
END = pd.Timestamp("2025-12-01")

print("Raíz del proyecto:", ROOT)

Raíz del proyecto: C:\Users\dra\Documents\brent-geopolitical-risk-predictor


## 2. Brent: carga, tipos y calidad

In [2]:
print("Hojas Brent:", pd.ExcelFile(BRENT_FILE).sheet_names)
brent_raw = pd.read_excel(BRENT_FILE, sheet_name="Data 1", header=2)

brent = brent_raw[[
    "Date",
    "Europe Brent Spot Price FOB (Dollars per Barrel)",
]].copy()
brent.columns = ["month", "brent_price"]
brent["month"] = pd.to_datetime(brent["month"], errors="coerce").dt.to_period("M").dt.to_timestamp()
brent["brent_price"] = pd.to_numeric(brent["brent_price"], errors="coerce")
brent = brent.dropna(subset=["month", "brent_price"]).sort_values("month").reset_index(drop=True)

print("Dimensiones originales:", brent_raw.shape)
print("Período disponible:", brent["month"].min().date(), "a", brent["month"].max().date())
print("Duplicados mensuales:", int(brent["month"].duplicated().sum()))
print("Faltantes:")
display(brent.isna().sum().to_frame("faltantes"))
display(brent.head())

assert not brent["month"].duplicated().any()

Hojas Brent: ['Contents', 'Data 1']
Dimensiones originales: (472, 2)
Período disponible: 1987-05-01 a 2026-08-01
Duplicados mensuales: 0
Faltantes:


,faltantes
month,0
brent_price,0


,month,brent_price
0,1987-05-01,18.58
1,1987-06-01,18.86
2,1987-07-01,19.86
3,1987-08-01,18.98
4,1987-09-01,18.31


## 3. GPR: carga, tipos y calidad

In [3]:
print("Hojas GPR:", pd.ExcelFile(GPR_FILE).sheet_names)
gpr_raw = pd.read_excel(GPR_FILE, sheet_name="Sheet1")

gpr = gpr_raw[["month", "GPR", "GPRT", "GPRA"]].copy()
gpr["month"] = pd.to_datetime(gpr["month"], errors="coerce").dt.to_period("M").dt.to_timestamp()
for column in ["GPR", "GPRT", "GPRA"]:
    gpr[column] = pd.to_numeric(gpr[column], errors="coerce")
gpr = gpr.dropna(subset=["month"]).sort_values("month").reset_index(drop=True)

print("Dimensiones originales:", gpr_raw.shape)
print("Inicio del GPR reciente:", gpr.loc[gpr["GPR"].notna(), "month"].min().date())
print("Duplicados mensuales:", int(gpr["month"].duplicated().sum()))
display(gpr[["GPR", "GPRT", "GPRA"]].isna().sum().to_frame("faltantes"))
display(gpr.head())

assert not gpr["month"].duplicated().any()

Hojas GPR: ['Sheet1']


Dimensiones originales: (1520, 115)
Inicio del GPR reciente: 1985-01-01
Duplicados mensuales: 0


,faltantes
GPR,1020
GPRT,1020
GPRA,1020


,month,GPR,GPRT,GPRA
0,1900-01-01,NaN,NaN,NaN
1,1900-02-01,NaN,NaN,NaN
2,1900-03-01,NaN,NaN,NaN
3,1900-04-01,NaN,NaN,NaN
4,1900-05-01,NaN,NaN,NaN


## 4. STEO: identificación de series y unidades

El CSV tiene cuatro filas de metadatos y luego una tabla horizontal. Se seleccionan las series por `source key`, no por posición. Si una serie aparece repetida, se comprueba que las filas mensuales sean idénticas antes de conservar una sola.

In [4]:
with STEO_FILE.open(encoding="utf-8-sig") as stream:
    steo_metadata_lines = [next(stream).strip() for _ in range(4)]

steo_raw = pd.read_csv(STEO_FILE, skiprows=4)
series_names = {
    "PAPR_WORLD": "world_production",
    "PATC_WORLD": "world_consumption",
    "COPR_WORLD": "world_crude_production",
}
month_columns = [
    column for column in steo_raw.columns
    if pd.notna(pd.to_datetime(column, format="%b %Y", errors="coerce"))
]

series_metadata = (
    steo_raw.loc[steo_raw["source key"].isin(series_names),
                 ["Unnamed: 1", "units", "source key"]]
    .drop_duplicates()
    .sort_values("source key")
)

print("Metadatos del archivo:")
for line in steo_metadata_lines:
    print(line)
print("\nSeries seleccionadas:")
display(series_metadata)
print("Filas por código:")
display(steo_raw["source key"].value_counts().reindex(series_names).fillna(0).astype(int).to_frame("filas"))

for key in series_names:
    rows = steo_raw.loc[steo_raw["source key"] == key, month_columns].apply(pd.to_numeric, errors="coerce")
    if rows.empty:
        raise ValueError(f"No se encontró la serie {key}")
    if rows.nunique(dropna=False).max() > 1:
        raise ValueError(f"Las filas repetidas de {key} no son idénticas")

assert set(series_metadata["units"]) == {"million barrels per day"}

Metadatos del archivo:
"3a. International Petroleum and Other Liquids Production, Consumption, and Inventories"
"https://www.eia.gov/outlooks/steo/data/browser/?utm_source=chatgpt.com#/?v=6&f=M&s=0&start=199801&end=202712&maptype=0&ctype=linechart&id=&linechart=PAPR_WORLD"
"Tue Sep 22 2026 11:31:48 GMT-0300 (Argentina Standard Time)"
"Source: U.S. Energy Information Administration"

Series seleccionadas:


,Unnamed: 1,units,source key
3,World Crude Oil Production,million barrels per day,COPR_WORLD
2,Total World Petroleum Production,million barrels per day,PAPR_WORLD
15,Total World Petroleum Consumption,million barrels per day,PATC_WORLD


Filas por código:


,filas
source key,
PAPR_WORLD,2
PATC_WORLD,1
COPR_WORLD,1


## 5. Transformación a frecuencia mensual y separación temporal

In [5]:
world_series = {}
for key, final_name in series_names.items():
    row = (
        steo_raw.loc[steo_raw["source key"] == key, month_columns]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
    )
    world_series[final_name] = row.to_numpy()

world_full = pd.DataFrame({
    "month": pd.to_datetime(month_columns, format="%b %Y"),
    **world_series,
}).sort_values("month").reset_index(drop=True)

world_full["world_market_balance"] = (
    world_full["world_production"] - world_full["world_consumption"]
)

outside_scope = world_full.loc[world_full["month"] > END]
world = world_full.loc[world_full["month"].between(START, END)].copy()

print("Cobertura STEO completa:", world_full["month"].min().date(), "a", world_full["month"].max().date())
print("Período histórico usado:", world["month"].min().date(), "a", world["month"].max().date())
print("Meses 2026-2027 excluidos del análisis:", len(outside_scope))
print("Faltantes en el período usado:")
display(world.isna().sum().to_frame("faltantes"))
display(world.head())

Cobertura STEO completa: 1998-01-01 a 2027-12-01
Período histórico usado: 1998-01-01 a 2025-12-01
Meses 2026-2027 excluidos del análisis: 24
Faltantes en el período usado:


,faltantes
month,0
world_production,0
world_consumption,0
world_crude_production,0
world_market_balance,0


,month,world_production,world_consumption,world_crude_production,world_market_balance
0,1998-01-01,76.49,73.08,66.56,3.41
1,1998-02-01,76.90,74.50,66.93,2.40
2,1998-03-01,76.71,74.30,66.81,2.41
3,1998-04-01,76.60,72.76,66.68,3.84
4,1998-05-01,75.97,70.74,66.14,5.23


## 6. Integración mensual y creación de objetivos

Se usa una unión interna en 1998-2025 y se valida una observación por mes. El precio siguiente, su retorno y la dirección se calculan después de ordenar la serie. Diciembre de 2025 queda sin objetivo porque enero de 2026 está fuera del alcance; no se inventa ese valor y el notebook de modelos excluirá esa única fila.

In [6]:
df = (
    brent.loc[brent["month"].between(START, END)]
    .merge(gpr.loc[gpr["month"].between(START, END)], on="month", how="inner")
    .merge(world, on="month", how="inner")
    .sort_values("month")
    .reset_index(drop=True)
)

expected_months = pd.date_range(START, END, freq="MS")
missing_months = expected_months.difference(df["month"])

df["brent_price_next_month"] = df["brent_price"].shift(-1)
df["brent_return_next_month"] = (
    df["brent_price_next_month"] / df["brent_price"] - 1
)
direction = np.where(
    df["brent_price_next_month"].notna(),
    (df["brent_price_next_month"] > df["brent_price"]).astype(int),
    np.nan,
)
df["brent_up_next_month"] = pd.Series(direction, dtype="Int64")
df["data_status"] = "historical_revised_STEO_vintage_2026_09"

final_columns = [
    "month", "brent_price", "GPR", "GPRT", "GPRA",
    "world_production", "world_consumption", "world_crude_production",
    "world_market_balance", "brent_price_next_month",
    "brent_return_next_month", "brent_up_next_month", "data_status",
]
df = df[final_columns]

print("Dimensiones integradas:", df.shape)
print("Período:", df["month"].min().date(), "a", df["month"].max().date())
print("Meses ausentes:", len(missing_months))
print("Duplicados:", int(df["month"].duplicated().sum()))
display(df.isna().sum().to_frame("faltantes"))
display(df.head())
display(df.tail())

assert len(df) == len(expected_months)
assert len(missing_months) == 0
assert not df["month"].duplicated().any()
assert df.loc[df.index[:-1], "brent_price_next_month"].notna().all()
assert df.loc[df.index[-1], "brent_price_next_month"] is np.nan or pd.isna(df.loc[df.index[-1], "brent_price_next_month"])

Dimensiones integradas: (336, 13)
Período: 1998-01-01 a 2025-12-01
Meses ausentes: 0
Duplicados: 0


,faltantes
month,0
brent_price,0
GPR,0
GPRT,0
GPRA,0
world_production,0
world_consumption,0
world_crude_production,0
world_market_balance,0
brent_price_next_month,1


,month,brent_price,GPR,GPRT,GPRA,world_production,world_consumption,world_crude_production,world_market_balance,brent_price_next_month,brent_return_next_month,brent_up_next_month,data_status
0,1998-01-01,15.19,53.096302,59.821270,42.487816,76.49,73.08,66.56,3.41,14.07,-0.073733,0,historical_revised_STEO_vintage_2026_09
1,1998-02-01,14.07,77.597618,101.840828,45.368019,76.90,74.50,66.93,2.40,13.10,-0.068941,0,historical_revised_STEO_vintage_2026_09
2,1998-03-01,13.10,57.248062,65.501999,44.516712,76.71,74.30,66.81,2.41,13.53,0.032824,1,historical_revised_STEO_vintage_2026_09
3,1998-04-01,13.53,50.996582,58.767384,38.002850,76.60,72.76,66.68,3.84,14.36,0.061345,1,historical_revised_STEO_vintage_2026_09
4,1998-05-01,14.36,94.473488,129.798889,43.802757,75.97,70.74,66.14,5.23,12.21,-0.149721,0,historical_revised_STEO_vintage_2026_09


,month,brent_price,GPR,GPRT,GPRA,world_production,world_consumption,world_crude_production,world_market_balance,brent_price_next_month,brent_return_next_month,brent_up_next_month,data_status
331,2025-08-01,67.87,138.988464,160.415192,121.731056,107.63,104.49,79.73,3.14,67.99,0.001768,1,historical_revised_STEO_vintage_2026_09
332,2025-09-01,67.99,125.869522,141.806107,114.789146,108.81,105.85,80.99,2.96,64.54,-0.050743,0,historical_revised_STEO_vintage_2026_09
333,2025-10-01,64.54,153.479935,166.623001,151.141785,108.55,104.44,80.78,4.11,63.80,-0.011466,0,historical_revised_STEO_vintage_2026_09
334,2025-11-01,63.80,103.939209,118.008568,88.702896,108.62,104.25,80.65,4.37,62.54,-0.019749,0,historical_revised_STEO_vintage_2026_09
335,2025-12-01,62.54,132.486221,146.358932,123.534622,107.97,106.20,80.54,1.77,NaN,NaN,<NA>,historical_revised_STEO_vintage_2026_09


## 7. Exportación reproducible

In [7]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False, date_format="%Y-%m")

print("Archivo guardado en:", OUTPUT_FILE)
print("Filas exportadas:", len(df))
print("Columnas exportadas:", len(df.columns))

Archivo guardado en: C:\Users\dra\Documents\brent-geopolitical-risk-predictor\data\processed\brent_gpr_market_monthly.csv
Filas exportadas: 336
Columnas exportadas: 13


## Conclusión

El resultado es una serie mensual completa de 336 meses. Brent, GPR y las variables mundiales comparten fecha y unidad temporal; el balance compara categorías compatibles; y los meses posteriores a 2025 quedan fuera. El dataset queda listo para el AED y para construir variables rezagadas sin usar información futura.